# LeetCode 1318 — Minimum Flips to Make a OR b Equal to c bằng Python

Link đề: https://leetcode.com/problems/minimum-flips-to-make-a-or-b-equal-to-c/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def minFlips(self, a: int, b: int, c: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nắm các phép toán bit trong Python: `&`, `|`, `^`, `>>`, `<<`, lấy bit thứ `i`, đếm bit 1 (`bin(x).count("1")`, `int.bit_count()`).
2. Hiểu nguyên lý **các bit độc lập nhau** với phép OR ⇒ chia bài toán thành 30 bài toán con 1 bit.
3. Lập **bảng quyết định cho từng bit** (`a_i`, `b_i`, `c_i`) → số lần lật tối thiểu.
4. Rút gọn thành **công thức bit-trick** `popcount((a|b)^c) + popcount(a & b & ((a|b)^c))`.
5. Tránh lỗi: đếm 1 lần lật khi cả hai bit đều là 1, dừng vòng lặp quá sớm, nhầm **độ ưu tiên** toán tử bit, so sánh `x & (1<<i) == 1`.

## 2. Đề bài & ví dụ

Cho 3 số nguyên dương `a`, `b`, `c`. Trả về **số lần lật bit ít nhất** trên `a` và `b` để `(a OR b) == c`. Một lần lật là đổi một bit `1 → 0` hoặc `0 → 1` trong biểu diễn nhị phân.

**Ví dụ 1:** `a = 2, b = 6, c = 5` → `3` (sau khi lật: `a = 1, b = 4`, `1 | 4 = 5`).

**Ví dụ 2:** `a = 4, b = 2, c = 7` → `1`.

**Ví dụ 3:** `a = 1, b = 2, c = 3` → `0` (vốn đã `1 | 2 = 3`).

**Ràng buộc:** `1 <= a, b, c <= 10^9`.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [((2, 6, 5), 3), ((4, 2, 7), 1), ((1, 2, 3), 0)]
for (a, b, c), want in examples:
    print(f"a={a:>2} ({a:04b}), b={b:>2} ({b:04b}), c={c:>2} ({c:04b}) -> mong đợi {want}")

a= 2 (0010), b= 6 (0110), c= 5 (0101) -> mong đợi 3
a= 4 (0100), b= 2 (0010), c= 7 (0111) -> mong đợi 1
a= 1 (0001), b= 2 (0010), c= 3 (0011) -> mong đợi 0


## 3. Kiến thức nền

### 3.1 Biểu diễn nhị phân

`bin(x)` trả chuỗi `'0b...'`; `format(x, '08b')` cho chuỗi đúng 8 ký tự (đệm 0 bên trái). `x.bit_length()` = số bit cần để biểu diễn `x`. Vì `10^9 < 2^30`, mọi số trong đề có tối đa **30 bit**.

In [2]:
x = 13
print(bin(x), format(x, "08b"), "| bit_length =", x.bit_length())
print("10^9 cần", (10**9).bit_length(), "bit; 2^30 =", 2**30)

0b1101 00001101 | bit_length = 4
10^9 cần 30 bit; 2^30 = 1073741824


### 3.2 Các phép toán bit

| Phép | Ý nghĩa từng bit | Ví dụ `6 (110)` và `5 (101)` |
|---|---|---|
| `&` (AND) | 1 nếu **cả hai** là 1 | `100` = 4 |
| `\|` (OR) | 1 nếu **ít nhất một** là 1 | `111` = 7 |
| `^` (XOR) | 1 nếu **khác nhau** | `011` = 3 |
| `>> k` | dịch phải k bit (bỏ k bit thấp) | `6 >> 1 = 3` |
| `<< k` | dịch trái k bit | `1 << 3 = 8` |

In [3]:
a, b = 6, 5
for name, v in [("a & b", a & b), ("a | b", a | b), ("a ^ b", a ^ b), ("a >> 1", a >> 1), ("1 << 3", 1 << 3)]:
    print(f"{name:7s} = {v:2d} ({v:04b})")

a & b   =  4 (0100)
a | b   =  7 (0111)
a ^ b   =  3 (0011)
a >> 1  =  3 (0011)
1 << 3  =  8 (1000)


### 3.3 Lấy bit thứ `i`

Hai cách tương đương: `(x >> i) & 1` (trả 0 hoặc 1) hoặc `x & (1 << i)` (trả 0 hoặc `2^i` — **không phải** 1!). Nên dùng cách đầu để so sánh với 0/1.

In [4]:
x = 0b1010
print("các bit từ thấp lên:", [(x >> i) & 1 for i in range(4)])
print("x & (1 << 3) =", x & (1 << 3), "(khác 0 nghĩa là bit 3 bật, nhưng không bằng 1)")

các bit từ thấp lên: [0, 1, 0, 1]
x & (1 << 3) = 8 (khác 0 nghĩa là bit 3 bật, nhưng không bằng 1)


### 3.4 Đếm số bit 1 (popcount)

`bin(x).count("1")` chạy mọi phiên bản Python; từ Python 3.10 có `x.bit_count()` nhanh hơn. Mẹo kinh điển `x &= x - 1` xoá bit 1 thấp nhất.

In [5]:
x = 0b101101
cnt, y = 0, x
while y:
    y &= y - 1                 # xoá bit 1 thấp nhất
    cnt += 1
print(bin(x).count("1"), x.bit_count(), cnt)

4 4 4


### 3.5 Độ ưu tiên toán tử bit trong Python

Từ cao xuống thấp: `<<`, `>>` > `&` > `^` > `|` > các phép so sánh (`==`, `<`…). Vì vậy `a | b ^ c` nghĩa là `a | (b ^ c)`, còn `(a | b) ^ c` phải có ngoặc. **Luôn viết ngoặc** khi trộn các phép bit.

In [6]:
a, b, c = 2, 6, 5
print("a | b ^ c   =", a | b ^ c, "(= a | (b ^ c))")
print("(a | b) ^ c =", (a | b) ^ c)

a | b ^ c   = 3 (= a | (b ^ c))
(a | b) ^ c = 3


### 3.6 XOR để tìm "chỗ khác nhau" (Hamming distance)

`x ^ y` có bit 1 đúng ở những vị trí `x` và `y` **khác nhau** ⇒ `popcount(x ^ y)` = số bit phải lật để biến `x` thành `y` (khoảng cách Hamming). Đây chính là cách đo "chi phí" giữa `a` và `a'` trong vét cạn, và là gốc của mặt nạ `(a | b) ^ c` ở lời giải tối ưu.

In [7]:
def hamming(x, y):
    return bin(x ^ y).count("1")

x, y = 0b1011, 0b0110
print(f"x={x:04b}, y={y:04b}, x^y={x ^ y:04b} -> cần lật {hamming(x, y)} bit")
print("lật xong:", format(x ^ (x ^ y), "04b"), "== y?", x ^ (x ^ y) == y)

x=1011, y=0110, x^y=1101 -> cần lật 3 bit
lật xong: 0110 == y? True


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `a = 2 (010)`, `b = 6 (110)`, `c = 5 (101)`. Xét từng cột bit (từ phải sang):

| bit i | a_i | b_i | c_i | a_i OR b_i | cần làm gì | số lần lật |
|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 1 | 0 | bật một trong hai bit | 1 |
| 1 | 1 | 1 | 0 | 1 | tắt **cả hai** bit | 2 |
| 2 | 0 | 1 | 1 | 1 | đã đúng | 0 |

Tổng `1 + 2 + 0 = 3` ✔.

Quan sát then chốt: OR tính **từng cột độc lập** — lật một bit ở cột `i` không ảnh hưởng cột khác. ⇒ Đáp án = tổng chi phí tối thiểu của từng cột.

In [8]:
a, b, c = 2, 6, 5
for i in range(3):
    ai, bi, ci = (a >> i) & 1, (b >> i) & 1, (c >> i) & 1
    cost = (1 if ai == 0 and bi == 0 else 0) if ci == 1 else ai + bi
    print(f"bit {i}: a={ai} b={bi} c={ci} -> lật {cost}")

bit 0: a=0 b=0 c=1 -> lật 1
bit 1: a=1 b=1 c=0 -> lật 2
bit 2: a=0 b=1 c=1 -> lật 0


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi cặp `(a', b')`

**Ý tưởng:** thử mọi cặp số `a'`, `b'` có cùng số bit, giữ những cặp có `a' | b' == c`, chi phí = số bit khác nhau giữa `a` và `a'` cộng giữa `b` và `b'` = `popcount(a ^ a') + popcount(b ^ b')`. Lấy nhỏ nhất.

Chỉ chạy được khi số bit `B` rất nhỏ: có `4^B` cặp — với `B = 30` là `10^18` ⇒ **bất khả thi**. Ta dùng nó để **kiểm chứng** trên số nhỏ.

In [9]:
def min_flips_brute(a, b, c):
    B = max(a.bit_length(), b.bit_length(), c.bit_length())
    best = None
    for a2 in range(1 << B):
        for b2 in range(1 << B):
            if a2 | b2 == c:
                cost = bin(a ^ a2).count("1") + bin(b ^ b2).count("1")
                if best is None or cost < best:
                    best = cost
    return best

for (a, b, c), want in examples:
    print((a, b, c), "->", min_flips_brute(a, b, c), "| mong đợi", want)

(2, 6, 5) -> 3 | mong đợi 3
(4, 2, 7) -> 1 | mong đợi 1
(1, 2, 3) -> 0 | mong đợi 0


**Độ phức tạp:** Time `O(4^B · B)` với `B` = số bit (≤ 30) ⇒ chỉ dùng cho `B <= 6`. Space `O(1)`.

**Nút thắt:** ta coi `a'`, `b'` như một khối, trong khi các bit **độc lập** — nên tối ưu từng bit riêng là đủ (`4` khả năng mỗi bit thay vì `4^B` tổ hợp).

In [10]:
for B in (6, 8, 10):
    a = b = c = (1 << B) - 1
    t0 = time.perf_counter()
    min_flips_brute(a, b, c)
    print(f"B={B}: {time.perf_counter() - t0:.3f}s  (mỗi +2 bit chậm ~16 lần)")

B=6: 0.000s  (mỗi +2 bit chậm ~16 lần)
B=8: 0.004s  (mỗi +2 bit chậm ~16 lần)
B=10: 0.076s  (mỗi +2 bit chậm ~16 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — độc lập theo bit:** bit `i` của `a | b` chỉ phụ thuộc `a_i`, `b_i`. Lật bit `i` không đổi bit khác ⇒ bài toán tách thành từng bit.

**Insight 2 — bảng quyết định cho 1 bit** (`a_i`, `b_i`, `c_i`):

| c_i | a_i, b_i | số lần lật tối thiểu | lý do |
|---|---|---|---|
| 1 | 0, 0 | **1** | bật một trong hai |
| 1 | có ít nhất một 1 | 0 | đã đúng |
| 0 | 0, 0 | 0 | đã đúng |
| 0 | đúng một bit 1 | 1 | tắt bit đó |
| 0 | 1, 1 | **2** | phải tắt cả hai |

Gọn lại: **nếu** `c_i = 1` **thì** lật `1` khi `a_i = b_i = 0`, ngược lại 0; **nếu** `c_i = 0` **thì** lật `a_i + b_i`.

**Insight 3 — công thức bit-trick:**
- `x = (a | b) ^ c` có bit 1 tại đúng những cột **sai** (cần ít nhất 1 lần lật).
- Trong các cột sai, cột nào cần **2** lần lật? Cột có `c_i = 0` và `a_i = b_i = 1` ⇔ bit 1 trong `a & b & x` (vì `a_i & b_i = 1` kéo theo `(a|b)_i = 1`, cột sai thì `c_i = 0`).
- ⇒ `đáp án = popcount(x) + popcount(a & b & x)`.

In [11]:
# Kiểm chứng bảng quyết định bằng vét cạn trên 1 bit
for ci, ai, bi in product((0, 1), repeat=3):
    brute = min((ai ^ a2) + (bi ^ b2) for a2 in (0, 1) for b2 in (0, 1) if (a2 | b2) == ci)
    rule = (1 if ai == 0 and bi == 0 else 0) if ci == 1 else ai + bi
    print(f"c={ci} a={ai} b={bi}: vét cạn={brute} quy tắc={rule}")
    assert brute == rule

c=0 a=0 b=0: vét cạn=0 quy tắc=0
c=0 a=0 b=1: vét cạn=1 quy tắc=1
c=0 a=1 b=0: vét cạn=1 quy tắc=1
c=0 a=1 b=1: vét cạn=2 quy tắc=2
c=1 a=0 b=0: vét cạn=1 quy tắc=1
c=1 a=0 b=1: vét cạn=0 quy tắc=0
c=1 a=1 b=0: vét cạn=0 quy tắc=0
c=1 a=1 b=1: vét cạn=0 quy tắc=0


## 7. Các cách tiếp theo

### 7.1 Cách 2 — So sánh từng ký tự của chuỗi nhị phân

Cách "dễ hình dung" nhất: đổi cả ba số sang chuỗi nhị phân cùng độ dài (đệm 0), rồi duyệt từng cột ký tự và áp dụng bảng quyết định.

In [12]:
def min_flips_strings(a, b, c):
    L = max(a.bit_length(), b.bit_length(), c.bit_length())
    sa, sb, sc = format(a, f"0{L}b"), format(b, f"0{L}b"), format(c, f"0{L}b")
    flips = 0
    for x, y, z in zip(sa, sb, sc):
        if z == "1":
            if x == "0" and y == "0":
                flips += 1
        else:
            flips += (x == "1") + (y == "1")
    return flips

for (a, b, c), want in examples:
    print((a, b, c), "->", min_flips_strings(a, b, c), "| mong đợi", want)

(2, 6, 5) -> 3 | mong đợi 3
(4, 2, 7) -> 1 | mong đợi 1
(1, 2, 3) -> 0 | mong đợi 0


Code nộp LeetCode (cách 2):

In [13]:
class SolutionStrings:
    def minFlips(self, a: int, b: int, c: int) -> int:
        L = max(a.bit_length(), b.bit_length(), c.bit_length())
        sa, sb, sc = format(a, f"0{L}b"), format(b, f"0{L}b"), format(c, f"0{L}b")
        flips = 0
        for x, y, z in zip(sa, sb, sc):
            if z == "1":
                flips += (x == "0" and y == "0")
            else:
                flips += (x == "1") + (y == "1")
        return flips

print(SolutionStrings().minFlips(2, 6, 5))

3


**Độ phức tạp cách 2:** Time `O(B)` với `B ≤ 30`, Space `O(B)` cho các chuỗi.

### 7.2 Cách 3 — Duyệt từng bit bằng dịch bit

Không cần chuỗi: lặp 31 lần (đủ cho `10^9`), lấy bit thấp nhất bằng `& 1`, rồi dịch phải cả ba số. Có thể dừng khi cả ba đều về 0.

In [14]:
def min_flips_bits(a, b, c):
    flips = 0
    while a or b or c:                   # còn bit nào khác 0 thì còn xét
        ai, bi, ci = a & 1, b & 1, c & 1
        if ci:
            flips += (ai | bi) ^ 1       # 1 nếu cả hai bằng 0
        else:
            flips += ai + bi
        a >>= 1; b >>= 1; c >>= 1
    return flips

for (a, b, c), want in examples:
    print((a, b, c), "->", min_flips_bits(a, b, c), "| mong đợi", want)

(2, 6, 5) -> 3 | mong đợi 3
(4, 2, 7) -> 1 | mong đợi 1
(1, 2, 3) -> 0 | mong đợi 0


Code nộp LeetCode (cách 3):

In [15]:
class SolutionBits:
    def minFlips(self, a: int, b: int, c: int) -> int:
        flips = 0
        for i in range(31):
            ai, bi, ci = (a >> i) & 1, (b >> i) & 1, (c >> i) & 1
            if ci:
                flips += 1 if ai == 0 and bi == 0 else 0
            else:
                flips += ai + bi
        return flips

print(SolutionBits().minFlips(4, 2, 7))

1


**Độ phức tạp cách 3:** Time `O(B)` = `O(31)` = `O(1)` theo ràng buộc, Space `O(1)`.

### 7.3 Cách 4 — Công thức bit-trick (không vòng lặp)

```python
x = (a | b) ^ c                     # các cột sai
ans = popcount(x) + popcount(a & b & x)   # cột sai cần 2 lần lật được cộng thêm 1
```

Từng đoạn:
- `(a | b) ^ c`: XOR với `c` để đánh dấu những cột mà OR hiện tại khác `c`.
- `a & b & x`: chỉ giữ cột sai mà **cả hai** bit đều 1 (khi đó `c_i` buộc phải là 0).
- `bin(...).count("1")` hoặc `.bit_count()` để đếm.

In [16]:
def min_flips_formula(a, b, c):
    x = (a | b) ^ c
    return bin(x).count("1") + bin(a & b & x).count("1")

for (a, b, c), want in examples:
    print((a, b, c), "->", min_flips_formula(a, b, c), "| mong đợi", want)

(2, 6, 5) -> 3 | mong đợi 3
(4, 2, 7) -> 1 | mong đợi 1
(1, 2, 3) -> 0 | mong đợi 0


Code nộp LeetCode (cách 4):

In [17]:
class SolutionFormula:
    def minFlips(self, a: int, b: int, c: int) -> int:
        x = (a | b) ^ c
        return x.bit_count() + (a & b & x).bit_count()

print(SolutionFormula().minFlips(1, 2, 3))

0


**Độ phức tạp cách 4:** Time `O(1)` (vài phép bit trên số ≤ 30 bit), Space `O(1)`. Đây là cách ngắn nhất; `int.bit_count()` cần Python 3.10+, nếu không có thì dùng `bin(x).count("1")`.

## 8. TRACE từng bước

Trace cách 3 (dịch bit) trên Ví dụ 1: `a = 2`, `b = 6`, `c = 5`.

| vòng | a (nhị phân) | b | c | a&1 | b&1 | c&1 | lật thêm | tổng |
|---|---|---|---|---|---|---|---|---|
| 1 | 010 | 110 | 101 | 0 | 0 | 1 | 1 | 1 |
| 2 | 01 | 11 | 10 | 1 | 1 | 0 | 2 | 3 |
| 3 | 0 | 1 | 1 | 0 | 1 | 1 | 0 | 3 |
| – | 0 | 0 | 0 | | | | dừng | **3** |

Trace cách 4: `a | b = 110`, `x = 110 ^ 101 = 011` (2 cột sai), `a & b & x = 010 & 110 & 011 = 010` (1 cột cần lật đôi) ⇒ `2 + 1 = 3`.

In [18]:
def trace_bits(a, b, c):
    flips, step = 0, 0
    while a or b or c:
        step += 1
        ai, bi, ci = a & 1, b & 1, c & 1
        add = ((ai | bi) ^ 1) if ci else ai + bi
        flips += add
        print(f"vòng {step}: a={a:b} b={b:b} c={c:b} | bit thấp a={ai} b={bi} c={ci} -> lật +{add}, tổng={flips}")
        a >>= 1; b >>= 1; c >>= 1
    print("=> đáp án:", flips)
    return flips

trace_bits(2, 6, 5)

vòng 1: a=10 b=110 c=101 | bit thấp a=0 b=0 c=1 -> lật +1, tổng=1
vòng 2: a=1 b=11 c=10 | bit thấp a=1 b=1 c=0 -> lật +2, tổng=3
vòng 3: a=0 b=1 c=1 | bit thấp a=0 b=1 c=1 -> lật +0, tổng=3
=> đáp án: 3


3

In [19]:
def trace_formula(a, b, c):
    W = max(a.bit_length(), b.bit_length(), c.bit_length(), 1)
    f = lambda v: format(v, f"0{W}b")
    x = (a | b) ^ c
    both = a & b & x
    print(f"a        = {f(a)}")
    print(f"b        = {f(b)}")
    print(f"a | b    = {f(a | b)}")
    print(f"c        = {f(c)}")
    print(f"x=(a|b)^c= {f(x)}  -> {bin(x).count('1')} cột sai")
    print(f"a & b & x= {f(both)}  -> {bin(both).count('1')} cột cần lật 2 lần")
    ans = bin(x).count("1") + bin(both).count("1")
    print("=> đáp án:", ans)
    return ans

trace_formula(2, 6, 5)
print()
trace_formula(4, 2, 7)

a        = 010
b        = 110
a | b    = 110
c        = 101
x=(a|b)^c= 011  -> 2 cột sai
a & b & x= 010  -> 1 cột cần lật 2 lần
=> đáp án: 3

a        = 100
b        = 010
a | b    = 110
c        = 111
x=(a|b)^c= 001  -> 1 cột sai
a & b & x= 000  -> 0 cột cần lật 2 lần
=> đáp án: 1


1

## 9. Code nộp LeetCode

Bản chính thức: duyệt từng bit (dễ giải thích, không phụ thuộc phiên bản Python). Công thức bit-trick ở cách 4 cũng được chấp nhận.

In [20]:
class Solution:
    def minFlips(self, a: int, b: int, c: int) -> int:
        flips = 0
        for i in range(31):                    # 10^9 < 2^30, 31 bit là dư
            ai = (a >> i) & 1
            bi = (b >> i) & 1
            ci = (c >> i) & 1
            if ci == 1:
                if ai == 0 and bi == 0:        # cần bật 1 bit
                    flips += 1
            else:
                flips += ai + bi               # tắt mọi bit 1 (có thể là 2)
        return flips

sol = Solution()
for (a, b, c), want in examples:
    got = sol.minFlips(a, b, c)
    print((a, b, c), "->", got, "✔" if got == want else "✘")

(2, 6, 5) -> 3 ✔
(4, 2, 7) -> 1 ✔
(1, 2, 3) -> 0 ✔


## 10. Vì sao thuật toán đúng

1. **Độc lập:** bit `i` của `a | b` bằng `a_i OR b_i`. Mọi lần lật chỉ đổi đúng một bit của `a` hoặc `b`. Do đó tập lần lật có thể chia theo cột, và điều kiện `a | b == c` ⇔ **mọi** cột `i` thoả `a_i' OR b_i' = c_i`. Chi phí tổng = tổng chi phí từng cột ⇒ tối thiểu tổng = tổng các tối thiểu.
2. **Tối ưu từng cột:** mỗi cột chỉ có 4 cấu hình `(a_i', b_i')`; bảng ở mục 6 liệt kê đúng giá trị nhỏ nhất (đã kiểm chứng bằng vét cạn 8 trường hợp).
3. **Công thức:** số cột sai = `popcount(x)`; mỗi cột sai tốn ít nhất 1, và tốn đúng 2 ⇔ `c_i = 0, a_i = b_i = 1` ⇔ bit `i` của `a & b & x` bằng 1. Cộng lại cho đúng tổng ở bước 1.
4. **Đủ bit:** `a, b, c <= 10^9 < 2^30` nên mọi bit từ 30 trở lên đều 0 ở cả ba số — duyệt 31 bit không bỏ sót.

## 11. Phân tích Time / Space

Gọi `B` = số bit của số lớn nhất (≤ 30).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn `(a', b')` | `O(4^B · B)` | `O(1)` | chỉ để kiểm chứng `B <= 6` |
| 2. Chuỗi nhị phân | `O(B)` | `O(B)` | dễ hình dung |
| 3. Dịch bit | `O(B)` | `O(1)` | bản nộp chính |
| 4. Công thức | `O(1)` (vài phép bit) | `O(1)` | ngắn nhất |

Với `B ≤ 30`, cả ba cách 2–4 đều là hằng số; khác biệt chỉ ở hằng số và độ dễ đọc.

## 12. Bộ test

Ví dụ đề, edge case (`a = b = c`, `c` có bit cao hơn `a`, `b`; `a`, `b` có bit cao hơn `c`; giá trị biên `10^9`), và đối chiếu: cách 1 (vét cạn) trên số ≤ 6 bit, các cách 2–4 với nhau trên số tới `10^9`.

In [21]:
ALL = [min_flips_strings, min_flips_bits, min_flips_formula,
       Solution().minFlips, SolutionStrings().minFlips, SolutionBits().minFlips, SolutionFormula().minFlips]
tests = [
    ((2, 6, 5), 3),
    ((4, 2, 7), 1),
    ((1, 2, 3), 0),
    ((7, 7, 7), 0),            # đã đúng
    ((1, 1, 8), 3),            # c có bit cao: bật bit 3, tắt bit 0 ở cả hai
    ((8, 1, 1), 1),            # a có bit cao hơn c: tắt bit 3 của a
    ((7, 7, 1), 4),            # bit 1, 2: mỗi bit lật 2 lần
    ((10**9, 10**9, 1), 2 * bin(10**9).count("1") + 1),   # 10^9 chẵn: tắt mọi bit 1 của cả hai, bật bit 0
    ((10**9, 1, 10**9 + 1), 0),
]
for (a, b, c), want in tests:
    for f in ALL:
        got = f(a, b, c)
        assert got == want, (f, a, b, c, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 9 test cố định qua cho 7 hàm


In [22]:
rng = random.Random(1318)
for _ in range(600):                                  # nhỏ: so với vét cạn
    a, b, c = (rng.randint(1, 63) for _ in range(3))
    want = min_flips_brute(a, b, c)
    for f in ALL:
        assert f(a, b, c) == want, (a, b, c)
for _ in range(20000):                                # lớn: các cách nhanh khớp nhau
    a, b, c = (rng.randint(1, 10**9) for _ in range(3))
    r = {f(a, b, c) for f in ALL}
    assert len(r) == 1, (a, b, c, r)
print("Vét cạn 600 bộ nhỏ + 20000 bộ lớn: khớp")

Vét cạn 600 bộ nhỏ + 20000 bộ lớn: khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm số cột sai (quên trường hợp phải lật 2 lần)

In [23]:
def wrong_count_mismatch(a, b, c):
    return bin((a | b) ^ c).count("1")          # SAI: cột c=0, a=b=1 cần 2 lần

print("sai :", wrong_count_mismatch(2, 6, 5), "| đúng:", Solution().minFlips(2, 6, 5))
print("sai :", wrong_count_mismatch(7, 7, 1), "| đúng:", Solution().minFlips(7, 7, 1))

sai : 2 | đúng: 3
sai : 2 | đúng: 4


**Vì sao sai:** ở bit 1 của Ví dụ 1 (`a_1 = b_1 = 1`, `c_1 = 0`) phải tắt **cả hai** bit. **Sửa:** cộng thêm `popcount(a & b & x)`, hoặc dùng `ai + bi` khi `c_i = 0`.

### Lỗi 2 — Dừng vòng lặp khi `c` hết bit

In [24]:
def wrong_stop_on_c(a, b, c):
    flips = 0
    while c:                                      # SAI: a, b có thể còn bit cao hơn c
        ai, bi, ci = a & 1, b & 1, c & 1
        flips += ((ai | bi) ^ 1) if ci else ai + bi
        a >>= 1; b >>= 1; c >>= 1
    return flips

print("sai :", wrong_stop_on_c(8, 1, 1), "| đúng:", Solution().minFlips(8, 1, 1))

sai : 0 | đúng: 1


**Vì sao sai:** `a = 8 (1000)`, `c = 1 (0001)`: sau 1 vòng `c = 0` nên dừng, bỏ qua bit 3 của `a` cần tắt. **Sửa:** lặp khi `a or b or c`, hoặc lặp cố định 31 bit.

### Lỗi 3 — Quên ngoặc: `a | b ^ c`

In [25]:
def wrong_precedence(a, b, c):
    x = a | b ^ c                                 # SAI: Python hiểu là a | (b ^ c)
    return bin(x).count("1") + bin(a & b & x).count("1")

for args in [(2, 6, 5), (1, 2, 3)]:
    print(args, "-> sai:", wrong_precedence(*args), "| đúng:", Solution().minFlips(*args))

(2, 6, 5) -> sai: 3 | đúng: 3
(1, 2, 3) -> sai: 1 | đúng: 0


**Vì sao sai:** trong Python `^` ưu tiên **cao hơn** `|`, nên `a | b ^ c = a | (b ^ c)`. Với `(1, 2, 3)`: `2 ^ 3 = 1`, `1 | 1 = 1` ⇒ báo 1 cột sai trong khi đáp án là 0. **Sửa:** `(a | b) ^ c`. Quy tắc: trộn phép bit thì **luôn** đặt ngoặc.

### Lỗi 4 — Lấy bit bằng `x & (1 << i) == 1`

In [26]:
def wrong_bit_test(a, b, c):
    flips = 0
    for i in range(31):
        ai = 1 if a & (1 << i) == 1 else 0        # SAI: a & (1<<i) bằng 2^i chứ không bằng 1
        bi = 1 if b & (1 << i) == 1 else 0
        ci = 1 if c & (1 << i) == 1 else 0
        flips += (1 if ai == 0 and bi == 0 else 0) if ci else ai + bi
    return flips

print("sai :", wrong_bit_test(2, 6, 5), "| đúng:", Solution().minFlips(2, 6, 5))
print("sai :", wrong_bit_test(8, 1, 1), "| đúng:", Solution().minFlips(8, 1, 1))

sai : 1 | đúng: 3
sai : 0 | đúng: 1


**Vì sao sai:** `a & (1 << i)` là `0` hoặc `2^i`; chỉ ở `i = 0` mới bằng 1. Mọi bit cao bị coi là 0 ⇒ chỉ bit 0 được xét. (Ngoài ra `&` ưu tiên cao hơn `==` nên biểu thức là `(a & (1<<i)) == 1` — đúng cú pháp nhưng sai ý.) **Sửa:** `(a >> i) & 1` hoặc `a & (1 << i) != 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có `AND / OR / XOR` giữa các số và hỏi "tối thiểu / đếm" ⇒ thử **tách theo từng bit** (các bit độc lập).
- Với mỗi bit, **liệt kê bảng 8 trường hợp** `(a_i, b_i, c_i)` — nhanh và không sót.
- Muốn bỏ vòng lặp: tìm **mặt nạ** (mask) đánh dấu cột cần xử lý (`(a|b)^c`), rồi `popcount`.
- `10^9 < 2^30`: duyệt 31 bit là an toàn; với `10^18` dùng 61–63 bit.
- Python: `x.bit_count()` (3.10+) hoặc `bin(x).count("1")`; **luôn ngoặc** khi trộn `& ^ |`.

## 15. Biến thể của bài

1. **`a AND b == c`**: bit `c_i = 1` cần cả hai là 1 ⇒ lật `(1−a_i) + (1−b_i)`; `c_i = 0` cần ít nhất một 0 ⇒ lật 1 nếu `a_i = b_i = 1` (bài tập 5).
2. **`a XOR b == c`**: mỗi cột sai chỉ cần **1** lần lật ⇒ `popcount(a ^ b ^ c)`.
3. **Chỉ được lật bit của `a`**: cột `c_i = 0, b_i = 1` thì **không thể** ⇒ trả `-1`.
4. **LeetCode 461 — Hamming Distance**: `popcount(x ^ y)`.
5. **LeetCode 2220 — Minimum Bit Flips to Convert Number**: cùng ý tưởng XOR + popcount.
6. **Nhiều số `a1 | a2 | … | ak == c`**: cột `c_i = 0` phải tắt mọi bit 1 (cộng số lượng); cột `c_i = 1` cần ≥ 1 bit 1.

In [27]:
# Biến thể 2 và 3
def min_flips_xor(a, b, c):
    return bin(a ^ b ^ c).count("1")

def min_flips_only_a(a, b, c):
    flips = 0
    for i in range(31):
        ai, bi, ci = (a >> i) & 1, (b >> i) & 1, (c >> i) & 1
        if ci == 0:
            if bi == 1:
                return -1              # không thể tắt bit của b
            flips += ai
        elif bi == 0 and ai == 0:
            flips += 1
    return flips

print("XOR:", min_flips_xor(2, 6, 5), "| chỉ lật a:", min_flips_only_a(2, 6, 5), min_flips_only_a(1, 4, 5))

XOR: 1 | chỉ lật a: -1 0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `(a', b')` | Thử mọi cặp số cùng số bit | `O(4^B·B)` | `O(1)` | Kiểm chứng số nhỏ |
| 2. Chuỗi nhị phân | So từng ký tự sau khi đệm 0 | `O(B)` | `O(B)` | Người mới, dễ hình dung |
| 3. Dịch bit | `(x >> i) & 1` + bảng quyết định | `O(B)` | `O(1)` | Bản nộp chuẩn, dễ giải thích |
| 4. Công thức | `popcount(x) + popcount(a&b&x)` | `O(1)` | `O(1)` | Ngắn nhất, khi đã hiểu rõ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minFlips(7, 7, 1)`, `Solution().minFlips(1, 1, 8)`, `Solution().minFlips(5, 3, 7)`?

**Bài 2 (tính tay):** Lập bảng từng bit cho `a = 9`, `b = 4`, `c = 6` và tính đáp án.

**Bài 3 (điền TODO):** Hoàn thành `bit_cost(ai, bi, ci)` trả số lần lật tối thiểu cho một cột.

**Bài 4 (tự viết lại):** Viết lại công thức bit-trick `my_min_flips(a, b, c)` không nhìn code.

**Bài 5 (mini challenge):** Viết `min_flips_and(a, b, c)`: số lần lật tối thiểu để `(a AND b) == c`. Kiểm chứng bằng vét cạn.

In [28]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def bit_cost_todo(ai, bi, ci):
    # TODO: c=1 -> 1 nếu a=b=0; c=0 -> a+b
    pass

def my_min_flips_todo(a, b, c):
    # TODO: x = (a | b) ^ c ; đếm x và a & b & x
    pass

def min_flips_and_todo(a, b, c):
    # TODO: duyệt từng bit với bảng quyết định cho AND
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `7 = 111`, `1 = 001`: bit 1, 2 đều cần tắt hai bit. (2) `1 | 1 = 1`, `c = 1000`: bật bit 3 (1 lần) và tắt bit 0 ở cả hai (2 lần). (3) `5 | 3 = 7` đã đúng.

**Bài 2:** (1) `9 = 1001`, `4 = 0100`, `6 = 0110`. (2) Bit 0: `c=0`, `a=1` ⇒ 1; bit 1: `c=1`, cả hai 0 ⇒ 1; bit 2: `c=1`, `b=1` ⇒ 0; bit 3: `c=0`, `a=1` ⇒ 1. (3) Tổng 3.

**Bài 3:** (1) Tách theo `ci`. (2) `ci == 1`: `int(ai == 0 and bi == 0)`. (3) `ci == 0`: `ai + bi`.

**Bài 4:** (1) Ngoặc `(a | b)`. (2) `x = (a | b) ^ c`. (3) Cộng hai popcount.

**Bài 5:** (1) `c_i = 1` ⇒ cần `a_i = b_i = 1` ⇒ lật `(1 − a_i) + (1 − b_i)`. (2) `c_i = 0` ⇒ cần ít nhất một bit 0 ⇒ lật 1 nếu `a_i = b_i = 1`, ngược lại 0. (3) Vét cạn: viết hàm giống `min_flips_brute` nhưng điều kiện `a2 & b2 == c`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
assert Solution().minFlips(7, 7, 1) == 4
assert Solution().minFlips(1, 1, 8) == 3
assert Solution().minFlips(5, 3, 7) == 0
# Bài 2
assert trace_bits(9, 4, 6) == 3

# Bài 3
def bit_cost(ai, bi, ci):
    if ci == 1:
        return 1 if ai == 0 and bi == 0 else 0
    return ai + bi

for ci, ai, bi in product((0, 1), repeat=3):
    assert bit_cost(ai, bi, ci) == min((ai ^ x) + (bi ^ y) for x in (0, 1) for y in (0, 1) if (x | y) == ci)
print("Bài 1–3: OK")

vòng 1: a=1001 b=100 c=110 | bit thấp a=1 b=0 c=0 -> lật +1, tổng=1
vòng 2: a=100 b=10 c=11 | bit thấp a=0 b=0 c=1 -> lật +1, tổng=2
vòng 3: a=10 b=1 c=1 | bit thấp a=0 b=1 c=1 -> lật +0, tổng=2
vòng 4: a=1 b=0 c=0 | bit thấp a=1 b=0 c=0 -> lật +1, tổng=3
=> đáp án: 3
Bài 1–3: OK


In [30]:
# Bài 4
def my_min_flips(a, b, c):
    x = (a | b) ^ c
    return bin(x).count("1") + bin(a & b & x).count("1")

# Bài 5
def min_flips_and(a, b, c):
    flips = 0
    for i in range(31):
        ai, bi, ci = (a >> i) & 1, (b >> i) & 1, (c >> i) & 1
        if ci == 1:
            flips += (1 - ai) + (1 - bi)
        elif ai == 1 and bi == 1:
            flips += 1
    return flips

def min_flips_and_brute(a, b, c):
    B = max(a.bit_length(), b.bit_length(), c.bit_length())
    return min(bin(a ^ a2).count("1") + bin(b ^ b2).count("1")
               for a2 in range(1 << B) for b2 in range(1 << B) if a2 & b2 == c)

rng = random.Random(42)
for _ in range(400):
    a, b, c = (rng.randint(1, 31) for _ in range(3))
    assert my_min_flips(a, b, c) == min_flips_brute(a, b, c)
    assert min_flips_and(a, b, c) == min_flips_and_brute(a, b, c), (a, b, c)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tách theo bit":**

```python
ans = 0
for i in range(31):                       # đủ cho giá trị <= 10^9
    ai, bi, ci = (a >> i) & 1, (b >> i) & 1, (c >> i) & 1
    ans += cost(ai, bi, ci)               # bảng quyết định 8 trường hợp
return ans
```

**Công thức bài 1318:**

```python
x = (a | b) ^ c
return bin(x).count("1") + bin(a & b & x).count("1")
```

**Checklist trước khi submit:**
- [ ] Cột `c_i = 0`, `a_i = b_i = 1` tính **2** lần lật.
- [ ] Duyệt đủ bit (không dừng khi chỉ `c` hết bit).
- [ ] Có ngoặc: `(a | b) ^ c`.
- [ ] Lấy bit bằng `(x >> i) & 1`, không so `x & (1 << i) == 1`.
- [ ] Đúng chữ ký: `minFlips(self, a: int, b: int, c: int) -> int`.